In [1]:
from langchain_openai import ChatOpenAI

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"),
                 model = "gpt-4o-mini")

In [3]:
llm.invoke("What is AI in 1 Line")

AIMessage(content='AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, particularly computer systems.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 14, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4018e77935', 'id': 'chatcmpl-EPU83LkU5C5uiTMWvD4A4OfgHfrZM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b4fa-a824-75c1-b38e-cd098c931f56-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 21, 'total_tokens': 35, 'input_token_det

In [3]:
from langchain_core.prompts import ChatPromptTemplate

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}"),
    ("human", "Explain {topic} to a {audience} in 3 lines")
])

In [6]:
format_msg = prompt.format_messages(domain="Banking", topic="AWS BedRock", audience="Senior Architects")    

In [7]:
response = llm.invoke(format_msg)

In [8]:
response.content

'AWS Bedrock is a fully managed service that simplifies the process of building and scaling generative AI applications by providing access to a variety of foundational models from leading AI companies. It offers developers the flexibility to customize these models without needing extensive machine learning expertise, utilizing an easy-to-use API. Bedrock seamlessly integrates with other AWS services, enabling data processing and deployment in a scalable cloud environment.'

In [9]:
from pydantic import BaseModel, Field

In [10]:
class CourseInfo(BaseModel):
    title: str = Field(description="Name of the Course")
    duration_weeks: int = Field(description="Duration of the course in weeks")
    topics: list[str] = Field(description="List of main topics covered in the course")


In [11]:
struct_llm = llm.with_structured_output(CourseInfo)


In [12]:
prompt = ChatPromptTemplate.from_template("Generate Course Plan for a {title} Corporate Training of 30 Hours of duration and 4 hours per day maximum with Implementation Approach & Custom Projects")

In [13]:
chain = prompt | struct_llm

In [14]:
result = chain.invoke({"title": "AWS Sagemaker & Bedrock"})

In [15]:
result.duration_weeks

2

In [16]:
result.title

'AWS SageMaker & Bedrock Corporate Training'

In [17]:
result.topics

['Introduction to AWS SageMaker',
 'Setting up AWS SageMaker Environment',
 'Data Preparation and Management',
 'Building Machine Learning Models with SageMaker',
 'Training Models with SageMaker',
 'Hyperparameter Tuning and Optimization',
 'Model Evaluation and Deployment',
 'Introduction to AWS Bedrock',
 'Using Bedrock for Large Language Models',
 'Fine-Tuning Models with Bedrock',
 'Integration of SageMaker and Bedrock',
 'Implementation of Custom Projects',
 'Best Practices and Case Studies']

In [18]:
result

CourseInfo(title='AWS SageMaker & Bedrock Corporate Training', duration_weeks=2, topics=['Introduction to AWS SageMaker', 'Setting up AWS SageMaker Environment', 'Data Preparation and Management', 'Building Machine Learning Models with SageMaker', 'Training Models with SageMaker', 'Hyperparameter Tuning and Optimization', 'Model Evaluation and Deployment', 'Introduction to AWS Bedrock', 'Using Bedrock for Large Language Models', 'Fine-Tuning Models with Bedrock', 'Integration of SageMaker and Bedrock', 'Implementation of Custom Projects', 'Best Practices and Case Studies'])

In [4]:
from langchain_core.tools import tool

In [5]:
@tool
def get_course_fee(course_name: str) -> str:
    """ get the fee for a BEPEC Course"""
    fees = {
        "data science": "INR 75000",
        "ai course": "INR 80000",
        "cloud computing": "INR 85000"
    }
    return fees.get(course_name.lower(), "Course not found")

In [6]:
@tool
def get_batch_dates(course_name: str) -> str:
    """ get the batch dates for a BEPEC Course"""
    batch_dates = {
        "data science": "1st Oct 2026",
        "ai course": "1st March 2027",
        "cloud computing": "1st Oct 2026"
    }
    return batch_dates.get(course_name.lower(), "Course not found"  )

In [7]:
llm_tools = llm.bind_tools([get_course_fee, get_batch_dates])

In [8]:
response = llm_tools.invoke("What is the fee for Data Science course and when is the next batch starting?")

In [12]:
response.tool_calls

[{'name': 'get_course_fee',
  'args': {'course_name': 'Data Science'},
  'id': 'call_TtbTxUQt3BCioujgk4izrTnn',
  'type': 'tool_call'},
 {'name': 'get_batch_dates',
  'args': {'course_name': 'Data Science'},
  'id': 'call_KFUrFiFD0AgFGRQhG3UE7n9E',
  'type': 'tool_call'}]

In [13]:
from langchain_core.messages import HumanMessage, ToolMessage

question = "What is the fee for Data Science course and when is the next batch starting?"

messages = [
    HumanMessage(content=question)
]


response = llm_tools.invoke(messages)

messages.append(response)

tools = {
    "get_course_fee": get_course_fee,
    "get_batch_dates": get_batch_dates
}

for tool_call in response.tool_calls:
    
    tool = tools[tool_call["name"]]
    
    tool_result = tool.invoke(tool_call["args"])

    messages.append(
        ToolMessage(
            content=str(tool_result),
            tool_call_id=tool_call["id"]
        )
    )

final_response = llm_tools.invoke(messages)

print(final_response.content)

The fee for the Data Science course is INR 75,000. The next batch is starting on 1st October 2026.


In [9]:
response.tool_calls

[{'name': 'get_course_fee',
  'args': {'course_name': 'Data Science'},
  'id': 'call_NnNRXQPeNm4SiysneXnZt4Ho',
  'type': 'tool_call'},
 {'name': 'get_batch_dates',
  'args': {'course_name': 'Data Science'},
  'id': 'call_4eqk80mIytdxdc25A8DB80YQ',
  'type': 'tool_call'}]

In [26]:
from langchain_community.document_loaders import TextLoader, WebBaseLoader

/var/folders/d_/47lp0l293sx5khzrnt9d5_2m0000gn/T/ipykernel_8014/1323405293.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [28]:
loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")
docs = loader.load()

In [29]:
print(f"Loaded {len(docs)} document(s)")
print(f"First 200 chars: {docs[0].page_content[:200]}")
print(f"Metadata: {docs[0].metadata}")

Loaded 1 document(s)
First 200 chars: LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main content
Metadata: {'source': 'https://python.langchain.com/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}


In [ ]:
#chunking

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [33]:
long_text = """
    LangChain is a framework for building applications powered by language models.
    It provides modular components like prompts, chains, agents, memory, and retrievers.
    With LangChain, you can build chatbots, RAG systems, AI agents, and much more.
    The framework supports multiple LLM providers including OpenAI, Anthropic, Google,
    and open-source models via Hugging Face and Ollama.
    """ * 5

In [34]:
long_text

'\n    LangChain is a framework for building applications powered by language models.\n    It provides modular components like prompts, chains, agents, memory, and retrievers.\n    With LangChain, you can build chatbots, RAG systems, AI agents, and much more.\n    The framework supports multiple LLM providers including OpenAI, Anthropic, Google,\n    and open-source models via Hugging Face and Ollama.\n    \n    LangChain is a framework for building applications powered by language models.\n    It provides modular components like prompts, chains, agents, memory, and retrievers.\n    With LangChain, you can build chatbots, RAG systems, AI agents, and much more.\n    The framework supports multiple LLM providers including OpenAI, Anthropic, Google,\n    and open-source models via Hugging Face and Ollama.\n    \n    LangChain is a framework for building applications powered by language models.\n    It provides modular components like prompts, chains, agents, memory, and retrievers.\n    W

In [35]:
splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,       # Max characters per chunk
        chunk_overlap=30,     # Overlap to preserve context
        separators=["\n\n", "\n", ".", " "]
    )

In [36]:
chunks = splitter.split_text(long_text)

In [37]:
chunks

['LangChain is a framework for building applications powered by language models.\n    It provides modular components like prompts, chains, agents, memory, and retrievers.',
 'With LangChain, you can build chatbots, RAG systems, AI agents, and much more.\n    The framework supports multiple LLM providers including OpenAI, Anthropic, Google,',
 'and open-source models via Hugging Face and Ollama.\n    \n    LangChain is a framework for building applications powered by language models.',
 'It provides modular components like prompts, chains, agents, memory, and retrievers.\n    With LangChain, you can build chatbots, RAG systems, AI agents, and much more.',
 'The framework supports multiple LLM providers including OpenAI, Anthropic, Google,\n    and open-source models via Hugging Face and Ollama.',
 'LangChain is a framework for building applications powered by language models.\n    It provides modular components like prompts, chains, agents, memory, and retrievers.',
 'With LangChain, yo

In [38]:
len(chunks)

13

In [39]:
for i, chunk in enumerate(chunks):
    print(i, chunk)

0 LangChain is a framework for building applications powered by language models.
    It provides modular components like prompts, chains, agents, memory, and retrievers.
1 With LangChain, you can build chatbots, RAG systems, AI agents, and much more.
    The framework supports multiple LLM providers including OpenAI, Anthropic, Google,
2 and open-source models via Hugging Face and Ollama.
    
    LangChain is a framework for building applications powered by language models.
3 It provides modular components like prompts, chains, agents, memory, and retrievers.
    With LangChain, you can build chatbots, RAG systems, AI agents, and much more.
4 The framework supports multiple LLM providers including OpenAI, Anthropic, Google,
    and open-source models via Hugging Face and Ollama.
5 LangChain is a framework for building applications powered by language models.
    It provides modular components like prompts, chains, agents, memory, and retrievers.
6 With LangChain, you can build chatbot

In [40]:
from langchain_openai import OpenAIEmbeddings

In [41]:
embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY") , 
                                  model="text-embedding-3-small")

In [42]:
query_vector = embeddings.embed_query("What is machine learning?")

In [43]:
len(query_vector)

1536

In [44]:
print(f"First 5 values: {query_vector[:5]}")

First 5 values: [-0.0024356842041015625, -0.0127410888671875, -0.00664520263671875, -0.03155517578125, 0.02874755859375]


In [46]:
docs = [
        "Machine learning is a subset of AI.",
        "Pizza is a popular Italian dish.",
        "Deep learning uses neural networks."
    ]

In [47]:
doc_vectors = embeddings.embed_documents(docs)
print(f"Embedded {len(doc_vectors)} documents")

Embedded 3 documents


In [48]:
from langchain_community.vectorstores import FAISS

In [49]:
from langchain_core.documents import Document

In [57]:
docs = [
        Document(page_content="BEPEC offers Data Science courses in Bangalore."),
        Document(page_content="LangChain helps build LLM applications quickly."),
        Document(page_content="Python is the most popular language for AI."),
        Document(page_content="RAG combines retrieval with text generation in Data Science"),
    ]

In [58]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [54]:
query = "where can i learn data science"

In [59]:
results = vectorstore.similarity_search(query, k=2)

In [60]:
results

[Document(id='5492e18e-cd02-4afd-8c5e-b0c8b9445982', metadata={}, page_content='BEPEC offers Data Science courses in Bangalore.'),
 Document(id='41d45e16-a82d-4c49-a24e-3adb8987f7ea', metadata={}, page_content='RAG combines retrieval with text generation in Data Science')]

In [81]:
docs = [
        Document(page_content="BEPEC Solutions was founded 13+ years ago in Bangalore."),
        Document(page_content="BEPEC has placed 30,000+ alumni across 500+ hiring partners."),
        Document(page_content="BEPEC Offers this Courses which include Data Analytics, Data Engineering, AI/ML, GenAI."),
        Document(page_content="BEPEC is headquartered in Koramangala, Bengaluru."),
    ]

In [82]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [83]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [64]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
 
Context: {context}
 
Question: {question}
 
Answer:""")

In [65]:
def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

In [67]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [84]:
rag = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

In [87]:
answer = rag.invoke("What courses BEPEC offers")

In [88]:
answer

'BEPEC offers courses in Data Analytics, Data Engineering, AI/ML, and GenAI.'